In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sym
from sympy import pprint
np.set_printoptions(linewidth=150)
import time

def pp(M, rounding=3, imag_tolerance = 1e-12):
    if type(M) is not np.ndarray:
        print(M)
    else:
        M = M.real.round(rounding) + 1j*M.imag.round(rounding)
        if imag_tolerance is not None:
            M = np.where(np.abs(M.imag) < imag_tolerance, M.real, M)
        print(np.round(M, rounding))

### Defining Class objects - complete graphs only

In [3]:
class QuantumSystem:
    def __init__(self, q, alpha = 'auto'):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        self.alpha = alpha

        self.psi0_a = self.superposition_state()
        self.zero_b = self.superposition_state(coeffs = [1] + [0]*(self.N-1))
        self.psi_kron_0 = np.kron(self.psi0_a, self.zero_b)
        
        self.I_a = np.eye(self.N)
        self.I_b = np.eye(self.N)
        self.Proj_ab = np.kron(self.I_a, self.zero_b @ self.zero_b.conj().T)    # projection operator - Identity on a, projection on |0> on b
        
        self.Phase = np.diag(self.phases(alpha))                      # phase operator from the phases, "U_a" in notes
        self.Swap = self.U_ab()                                                 # swap operator, "U_ab" in notes

        self.U_big = self.Swap @ (np.kron(self.Phase, self.I_b)) @ self.Swap      # full unitary matrix
        
    def matrix_basis(self): 
        q = self.q
        N = 2**q
        basis = []
        for i in range(N):
            state_i = np.zeros(N)
            state_i[i] = 1
            basis.append(state_i)
        return np.array(basis)

    def superposition_state(self, coeffs = 'maximal_except_0'):        # careful, dont include the zeros
        q = self.q
        N = self.N
        if coeffs == 'maximal_except_0':
            coeffs = np.ones(N)
            coeffs[0] = 0

        basis = self.matrix_basis()

        c = np.array(coeffs).reshape(1, N)
        c = c/np.sqrt(np.sum(np.abs(c)**2))  # normalizing 

        if basis.shape[1] != c.shape[1]:
            raise ValueError('Error: Shapes of coefficients and basis do not match')
        if len(coeffs) != N:            # checking length
            raise ValueError('Error: The number of coefficients is not 2^q')

        sup_state = np.sum(c * basis, axis = 1, keepdims = True)
        if (np.linalg.norm(sup_state)-1) > 1e-10:
            raise ValueError('Error: The resultant state is not normalized')
        return sup_state
    
    
    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U
    
    # def manual_cross_edges(self, adjacency_M, alpha = 'auto'):

    def phases(self, alpha = 'auto'):       # phases for the phase operator
        q, N = self.q, self.N
        # alpha is the coefficient in the Hamiltonian
        if q%2 == 0:            # creating alpha automatically, see notebook
            alpha = 4*np.pi/q**2
        else:
            alpha = 4*np.pi/(q**2-1)
        if alpha != 'auto':
            alpha = alpha           

        def qubit_sequence(a,q):                # to be used for creating a particular qubit sequence/permutation
            b = np.binary_repr(a, width = q)       # string     
            c = np.array([int(i) for i in b])   # array
            return c

        def edges_count(seq):   #seq is a binary sequence array
            p_0 = np.count_nonzero(seq == 0)
            p_1 = np.count_nonzero(seq == 1)
            return int(p_0*p_1)                 # for fully connected graph

        phases = []
        for i in range(N):
            cross_edges_i = edges_count(qubit_sequence(i, q))           # qubit sequence is creating a binary permutation (qubit states permutations) from the integer i from 0 to N-1.
            phase_i = np.exp(-1j*alpha*cross_edges_i)
            phases.append(phase_i)
        return np.array(phases)


    @staticmethod
    def norm_sq(state):        # returns sum of absolute squares ie. sum of probabilities
        return (np.abs(state)**2).sum()
    
    @staticmethod
    def normalize(state):
        if type(state) is not np.ndarray:
            raise ValueError('Error: State is not a numpy array')
        return (1/np.sqrt(QuantumSystem.norm_sq(state)))*state

        


### Defining Class for General graphs

In [4]:
# defining class for the general graph as q+q qubit quantum system.
# to take in adjacency matrix, number of qubits. Will have methods to calculate the phase operator, the swap operator, the full unitary matrix, the superposition state, the measurement operator etc.


class QuantumSystem:
    def __init__(self, q, adj_matrix):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        self.basis_states = np.array([np.eye(self.N)[i] for i in range(self.N)])        # basis states for the q-qubits, list of N = 2^q states
        
        self.adj_matrix = adj_matrix
        self.E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph
        self.alpha = -np.pi/(2*self.E_t)

        self.psi0 = self.superposition_state()                              # homogeneous superposition state in q system qubits, 2^q dim.
        self.zero = self.superposition_state(coeffs = [1] + [0]*(self.N-1))  # |0> state for q anicillas
        self.psi0_tensor_zero = np.kron(self.psi0, self.zero)                  # to be sent in, 2^(2q) dim.
        
        self.I_a, self.I_b = np.eye(self.N), np.eye(self.N)                     
    

        start_time = time.time()
        self.Swap = self.U_ab()                                                 # swap operator, "U_ab" in notes
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f'Elapsed time for q = {q}:', elapsed_time)    
        
        self.Proj_ab = np.kron(self.I_a, self.zero @ self.zero.conj().T)    # projection operator - Identity on the system, projection on |0> state of ancillas
        self.phase_list = self.phases(self.q, self.adj_matrix)      # list of phases ANGLES
        self.Phase = np.diag(np.exp(1j*self.phase_list))                      # phase operator from the phases, "U_a" in notes
    
        self.U_big = self.Swap @ (np.kron(self.Phase, self.I_b)) @ self.Swap      # full unitary matrix

    # instead of defining a method for creating matrix basis we will do it as a class variable
    
    def superposition_state(self, coeffs = 'maximal_except_0'):        # careful, dont include the zeros
        q = self.q
        N = self.N
        if coeffs == 'maximal_except_0':
            coeffs = np.ones(N)
            coeffs[0] = 0

        c = np.array(coeffs).reshape(1, N)
        c = c/np.sqrt(np.sum(np.abs(c)**2))

        if self.basis_states.shape[1] != c.shape[1]:
            raise ValueError('Error: Shapes of coefficients and basis do not match')
        if len(coeffs) != N:            # checking length
            raise ValueError('Error: The number of coefficients is not 2^q')
        
        sup_state = np.sum(c * self.basis_states, axis = 1, keepdims = True)
        if (np.linalg.norm(sup_state)-1) > 1e-10:
            raise ValueError('Error: The resultant state is not normalized')
        return sup_state

    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U


    # defining the phase operator after defining the phi_x function for an phase of an individual permutation/basis state

    @staticmethod
    def qubit_sequence(x, q):
        b = np.binary_repr(x, width = q)       # binary string for the integer/basis state x (out of 2**q states) 
        c = np.array([int(i) for i in b])       # eg. '101' -> [1, 0, 1]
        return c

    # permut_arr is the array for the basis state x (which is a permutation of q qubits)

    @staticmethod
    def cross_edges_max_arr(permut_arr):             #all possible cross edges for this permutation (will add graph adjacency matrix later)
        permut_arr[permut_arr == 0] = -1            # calling the zeroes as -1, for distinguishing the cross edges upon multiplication
        col_indices, row_indices = np.meshgrid(permut_arr, permut_arr)  # eg. for permutation 101, col_indices will have columns 1,-1 and 1; row_indices will have rows 1,-1 and 1.

        CE_raw = col_indices * row_indices      # cross edges are -1, rest are 1
        CE_raw = np.triu(CE_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
        CE_raw[CE_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
        cross_edge_arr = (-1)*CE_raw                       # changing cross edges from -1 to 1
        return cross_edge_arr                    # returns upper triangular array of 1 and 0s, 1 for cross edge, 0 for self edge or no edge


    def phi_x(self, x, q, adj):                 # returns the angle phi for the basis state/partition x
        """ Returns phi and alpha for the particular basis state/partition x, given the adjacency matrix of the graph."""
        partition_string = self.qubit_sequence(x, q)
        partition_string_copy = partition_string.copy()

        C_E_max_arr_x = self.cross_edges_max_arr(partition_string_copy)
        total_edges = np.sum(adj) / 2  # adj is the adjacency matrix of the given graph
        C_E_actual = C_E_max_arr_x * adj
        C_E_count = np.sum(C_E_actual)
        alpha = -np.pi / (2 * total_edges)
        phi = -2 * alpha * C_E_count
        return phi


    def phases(self, q, adj_matrix):        # runs phi_x for all basis states/partitions; returns the list of PHASES (angles, NOT exponential of them) for the whole basis
        N = 2**q
        E_t = np.sum(adj_matrix)/2      # total number of edges in the given graph 

        phase_list = []
        for i in range(N):
            phase_list.append(self.phi_x(i, q, adj_matrix))
        phases = np.array(phase_list)
        return phases

    @staticmethod
    def norm_sq(state):        # returns sum of absolute squares ie. sum of probabilities
        return (np.abs(state)**2).sum()
    
    @staticmethod
    def normalize(state):
        if type(state) is not np.ndarray:
            raise ValueError('Error: State is not a numpy array')
        return (1/np.sqrt(QuantumSystem.norm_sq(state)))*state
    
    # @staticmethod
    # def tensor_div(arr,b):            # arr is tenosr prod of two states, b is the second state with all zeroes except one 1 at b'th position
    #     arr_copy = arr.copy()
    # @staticmethod
    # def state_to_qubit:


In [4]:
# create a function to iterate as before, to ultimately get bit strings with the highest probabilities
def iterate_system(q, adj_matrix, iterations = 3):
    start_time = time.time()
    
    Q = QuantumSystem(q, adj_matrix)
    psi = Q.psi0_tensor_zero
    U_big = Q.U_big
    prob_list, psi_iteration_list, psi_prime_list = [], [], []
    unnormalized_psi_list = []

    for i in range(iterations):
        psi_prime = U_big @ psi  # operated state
        psi_prime_list.append(psi_prime)

        psi_collapsed_unnorm = Q.Proj_ab @ psi_prime
        
        unnormalized_psi_list.append(psi_collapsed_unnorm)

        prob = Q.norm_sq(psi_collapsed_unnorm)
        prob_list.append(prob)
        psi = Q.normalize(psi_collapsed_unnorm)
        psi_iteration_list.append(psi)
        
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f'Elapsed time for q = {q}:', elapsed_time)
    return prob_list, psi,  psi_iteration_list, unnormalized_psi_list, Q.psi0_tensor_zero

In [12]:
# adj matrix for q = 8, linear

q = 8
adj_matrix = np.array([[0, 1, 0, 0, 0, 0, 0, 0],  [1, 0, 1, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 0, 0, 0], [0, 0, 1, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 1, 0, 0], [0, 0, 0, 0, 1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 1], [0, 0, 0, 0, 0, 0, 1, 0]])


# q = 4
# adj_matrix = np.array([[0, 1, 1, 0], 
#                          [1, 0, 1, 1],
#                         [1, 1, 0, 1],
#                         [0, 1, 1, 0]])

In [13]:
Q = QuantumSystem(q, adj_matrix)

Elapsed time for q = 8: 2130.7434911727905


: 